# Déploiement sur Cloud Run

Cloud Run exécute le conteneur sans administrer une VM. La facturation suit le temps d'exécution.
Prérequis cloud : image déjà poussée au registre, connecteur VPC vers le réseau de la VM MLflow, variables d'environnement du conteneur.
Le port d'écoute du conteneur est 80. L'URL du service remplace l'adresse IP éphémère de la VM.


## Serverless

Le fournisseur provisionne, répartit la charge et maintient l'exécution. Cloud Run lance des conteneurs. Cloud Functions lance du code (Python, Node.js, Go, etc.).
Chaque fonction est indépendante et sans état : rien n'est conservé d'un appel à l'autre. Les données partagées vont dans une base.
Le cold start est le délai de chargement en mémoire lorsqu'aucune requête récente n'a maintenu la fonction active.


## Connecteur VPC

Cloud Run n'est pas dans le VPC du projet. Le nom d'hôte interne de MLflow n'y est pas résolu, comme un ping lancé hors du projet.
Créer un accès VPC sans serveur. Sa plage d'adresses ne doit pas chevaucher les plages déjà attribuées (liste des réseaux VPC).
La plage proposée par défaut dans le cours est `10.8.0.0/28`.


## Service Cloud Run

Le nom du service et la région ne sont plus modifiables ensuite. Chaque déploiement crée une révision.
Conserver le port 80, allouer les ressources, puis saisir les mêmes variables d'environnement que pour le conteneur sur la VM.
Dans Connexions, associer le connecteur VPC. Autoriser tout le trafic entrant. L'authentification des appelants pourra être ajoutée sur cette API ensuite.
Le service expose une URL stable, sans adresse IP à suivre.


## Test


In [ ]:
import os
import requests
import pandas as pd

dataset = pd.read_csv(os.path.expanduser("data/primary.csv"))
dataset = dataset.drop(["user_session", "user_id", "purchased"], axis=1)


In [ ]:
requests.post(
    "https://purchase-predict-api-808424404935.us-central1.run.app/predict",  # Remplacer par l'URL du service Cloud Run
    json=dataset.sample(n=10).to_json()
).json()


Les requêtes HTTP apparaissent dans l'onglet Journaux du service.
